In [2]:
#download the pretrained model
# -*- coding: utf-8 -*-
from huggingface_hub import snapshot_download
snapshot_download(repo_id="m3rg-iitd/matscibert", local_dir="./matscibert")

/opt/anaconda3/envs/bmg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
.gitattributes: 100%|██████████| 1.23k/1.23k [00:00<00:00, 10.1MB/s]
config.json: 100%|██████████| 620/620 [00:00<00:00, 4.70MB/s]t]
README.md: 100%|██████████| 1.56k/1.56k [00:00<00:00, 18.0MB/s]
special_tokens_map.json: 100%|██████████| 112/112 [00:00<00:00, 895kB/s]








vocab.txt: 100%|██████████| 228k/228k [00:00<00:00, 721kB/s]
tokenizer.json: 100%|██████████| 467k/467k [00:02<00:00, 227kB/s]



































































































































































model.safetensors: 100%|██████████| 440M/440M [01:14<00:00, 5.87MB/s]
Fetching 9 files:  44%|████▍     | 4/9 [01:17<01:43, 20.74s/it]




pytorch_model.bin: 100%|██████████| 

'/Users/siyuliu/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/Project/BMG/MgBERT_LLM_Classification_for_Materials_Science/matscibert'

In [3]:
from tokenizers.normalizers import BertNormalizer


f = open('vocab_mappings.txt', 'r')
mappings = f.read().strip().split('\n')
f.close()

mappings = {m[0]: m[2:] for m in mappings}

norm = BertNormalizer(lowercase=False, strip_accents=True, clean_text=True, handle_chinese_chars=True)

def normalize(text):
    text = [norm.normalize_str(s) for s in text.split('\n')]
    out = []
    for s in text:
        norm_s = ''
        for c in s:
            norm_s += mappings.get(c, ' ')
        out.append(norm_s)
    return '\n'.join(out)

In [4]:
import torch
from torch import nn
import random
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Subset
from transformers import AutoModel, AutoTokenizer, AutoConfig


def setup_seed(seed):
     torch.manual_seed(seed)
     torch.cuda.manual_seed_all(seed)
     np.random.seed(seed)
     random.seed(seed)
## Set random seed
setup_seed(42)

config = AutoConfig.from_pretrained('./matscibert')
config.max_position_embeddings = 900
bert_model = AutoModel.from_pretrained('./matscibert', config=config, ignore_mismatched_sizes=True)


class BertClassifier(nn.Module):
    def __init__(self, dropout=0.5):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(768, 3)
        self.relu = nn.ReLU()

    def forward(self, input_id, mask):
        outputs = self.bert(input_ids=input_id, attention_mask=mask,return_dict=True, output_attentions=True)
        pooled_output = outputs.pooler_output
        attentions = outputs.attentions
        dropout_output = self.dropout(pooled_output)
        linear_output = self.linear(dropout_output)
        final_layer = self.relu(linear_output)
        return final_layer, attentions


## Set tokenizer
tokenizer = AutoTokenizer.from_pretrained('./matscibert')
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

## Load model
from torch.serialization import load
model_path = 'MgBERT.pth'
model_data = torch.load(model_path, map_location=device)
model = BertClassifier()
model.to(device)
model.load_state_dict(model_data)
model.eval()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
Some weights of BertModel were not initialized from the model checkpoint at ./matscibert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at ./matscibert and are newly initialized because the shapes did not match:
- bert.embeddings.position_embeddings.weight: found shape torch.Size([512, 768]) in the checkpoint and torch.Size([900, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(900, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

In [5]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('multi_label_examples.csv')

# Extract labels and compositions
labels = df['glass_forming_category'].tolist()
compositions = df['composition'].tolist()


In [6]:
def find_text(folder_path, id):
    file_path = os.path.join(folder_path, f'{id}.txt')
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} not found.")
    with open(file_path, 'r') as file:
        text = file.read()
    return text

In [7]:
def inference(input_text):
    inputs = tokenizer(normalize(input_text),
                                padding='max_length', 
                                max_length = 900, 
                                truncation=True,
                                return_tensors="pt").to(device)
    output, attention = model(inputs['input_ids'], inputs['attention_mask'])
    return output.argmax(dim=1)

In [8]:
results = []
for i in range(len(compositions)):
    composition = compositions[i]
    input_text = find_text('./llm/description/', composition)
    label = labels[i]
    result = inference(input_text)
    results.append([composition, label, result])

for composition, label, result in results:
    print(f"Composition: {composition}, Label: {label}")
    if result == 0 and label == 'BMG':
        print(f"The composition {composition} is a bulk metallic glass.")
    elif result == 1 and label == 'Ribbon':
        print(f"The composition {composition} is a ribbon-like metallic glass.")
    elif result == 2 and label == 'NR':
        print(f"The composition {composition} is not a metallic glass.")
    else:
        print(f"The composition {composition} is not classified correctly.")

Composition: Fe72Al5Ga2P10C6B4Si1, Label: BMG
The composition Fe72Al5Ga2P10C6B4Si1 is a bulk metallic glass.
Composition: Fe61Co7Zr10Mo5W2B15, Label: BMG
The composition Fe61Co7Zr10Mo5W2B15 is a bulk metallic glass.
Composition: Ca63Al32Cu5, Label: BMG
The composition Ca63Al32Cu5 is a bulk metallic glass.
Composition: Fe67W5Y6B22, Label: BMG
The composition Fe67W5Y6B22 is a bulk metallic glass.
Composition: Mg67.5Pd17.5Yb15, Label: BMG
The composition Mg67.5Pd17.5Yb15 is a bulk metallic glass.
Composition: Ag53.8Mg15.4Ca23.1Cu7.7, Label: BMG
The composition Ag53.8Mg15.4Ca23.1Cu7.7 is a bulk metallic glass.
Composition: Pd4Zr48Cu32Al8Ag8, Label: BMG
The composition Pd4Zr48Cu32Al8Ag8 is a bulk metallic glass.
Composition: Zr41Ti14Cu12.5Ni8Be22.5Fe2, Label: BMG
The composition Zr41Ti14Cu12.5Ni8Be22.5Fe2 is a bulk metallic glass.
Composition: Gd60Co25Al15, Label: BMG
The composition Gd60Co25Al15 is a bulk metallic glass.
Composition: Ni40Cu5Ti16.5Zr28.5Al10, Label: BMG
The composition Ni40